# Neighborhood Multi-Metric Ranking Prototype

This notebook contains only the standalone neighborhood ranking table prototype.

It compares and can rank Seattle MCPP neighborhoods by:

- **Response Time**
- **Call Volume**
- **Crime Volume**

The notebook uses the production crime and CAD/calls contexts and keeps the table methodology isolated from the rest of the v1.1 figure workbook.


In [ ]:
from pathlib import Path
import sys

# Locate repository root whether Jupyter starts from the repo root
# or from the notebooks directory.
cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected to find app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")


In [ ]:
import pandas as pd

from dash import Dash, Input, Output, State, dcc, html

from dashboard.analysis_windows import (
    get_analysis_bounds,
    get_history_bounds,
)
from dashboard.crime_classification import (
    CANONICAL_CRIME_TYPES,
)
from dashboard.crime_controls import (
    make_analysis_controls,
    make_analysis_state,
    make_neighborhood_options,
    validate_analysis_dates,
)
from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
)
from dashboard.spd_dashboard_data import (
    load_dashboard_context,
)

crime_context = load_crime_dashboard_context()
calls_context = load_dashboard_context()

crime = crime_context["valid_time"].copy()

history_start, history_end = get_history_bounds(
    crime["offense_date"]
)

analysis_start, analysis_end = get_analysis_bounds(
    history_end
)

analysis_start = analysis_start.date().isoformat()
analysis_end = analysis_end.date().isoformat()

default_start = history_end.date().isoformat()
default_end = history_end.date().isoformat()

default_categories = list(CANONICAL_CRIME_TYPES)

crime_category_options = [
    {"label": category.title(), "value": category}
    for category in default_categories
]

crime_subcategory_options = [
    {"label": value.title(), "value": value}
    for value in (
        crime["offense_sub_category"]
        .dropna()
        .astype("string")
        .str.strip()
        .sort_values()
        .unique()
        .tolist()
    )
    if value not in ["", "nan", "none"]
]

crime_neighborhood_options = make_neighborhood_options(
    crime["mcpp_neighborhood"]
)

RESPONSE_PRIORITY_OPTIONS = {
    "Priority 1–3": [1, 2, 3],
    "Priority 1–2": [1, 2],
    "Priority 1 only": [1],
}


## 10. Neighborhood multi-metric ranking (Assignee: Ben Carr)

This prototype combines three neighborhood measures in one ranking table:

- **Response Time** — median `response_time_minutes` from `calls_context["response_analysis"]`, qualified by the local priority selector.
- **Call Volume** — distinct `cad_event_number` from `calls_context["valid_time"]`.
- **Crime Volume** — distinct `offense_id` from `crime_context["valid_time"]`.

**Geography:** the table uses Seattle's canonical MCPP neighborhood vocabulary. Calls/response records use normalized `dispatch_neighborhood`; crime uses analytical `mcpp_neighborhood`. Unknown/non-canonical call neighborhoods are excluded rather than invented or reassigned.

**Ranking behavior:** the local **Rank by** selector controls ordering. Rank 1 means the largest selected metric: slowest median response time, highest call volume, or highest crime volume. The minimum qualified-response-event threshold applies only when ranking by Response Time.

**Period behavior:** all three columns use one shared effective calendar period, clamped to overlapping crime/CAD coverage so the displayed metrics refer to the same dates. Crime type, crime subcategory, and neighborhood selections do not filter this prototype; only the selected dates are shared.

> Tie policy and the final production label for this combined table remain methodology/design decisions.


In [ ]:
from dashboard.crime_dashboard_data import (
    normalize_neighborhood_name,
)
from dashboard.spd_config import (
    EVENT_ID_COLUMN as CALL_EVENT_ID_COLUMN,
    TIME_COLUMN as CALL_TIME_COLUMN,
)


ranking_response = (
    calls_context["response_analysis"]
    .copy()
)
ranking_calls = (
    calls_context["valid_time"]
    .copy()
)
ranking_crime = (
    crime_context["valid_time"]
    .copy()
)

ranking_response["queued_time"] = pd.to_datetime(
    ranking_response["queued_time"],
    errors="coerce",
)
ranking_calls[CALL_TIME_COLUMN] = pd.to_datetime(
    ranking_calls[CALL_TIME_COLUMN],
    errors="coerce",
)
ranking_crime["offense_date"] = pd.to_datetime(
    ranking_crime["offense_date"],
    errors="coerce",
)

# Canonical row vocabulary comes from the production MCPP boundaries.
canonical_ranking_neighborhoods = (
    crime_context["mcpp_boundaries"]
    [["mcpp_neighborhood"]]
    .copy()
)
canonical_ranking_neighborhoods[
    "mcpp_neighborhood"
] = normalize_neighborhood_name(
    canonical_ranking_neighborhoods[
        "mcpp_neighborhood"
    ]
)
canonical_ranking_neighborhoods = (
    canonical_ranking_neighborhoods
    .dropna()
    .drop_duplicates()
    .sort_values("mcpp_neighborhood")
    .reset_index(drop=True)
)
canonical_ranking_set = set(
    canonical_ranking_neighborhoods[
        "mcpp_neighborhood"
    ].astype(str)
)

# Normalize source geography only for matching to the canonical vocabulary.
ranking_response["ranking_neighborhood"] = (
    normalize_neighborhood_name(
        ranking_response["dispatch_neighborhood"]
    )
)
ranking_calls["ranking_neighborhood"] = (
    normalize_neighborhood_name(
        ranking_calls["dispatch_neighborhood"]
    )
)
ranking_crime["ranking_neighborhood"] = (
    normalize_neighborhood_name(
        ranking_crime["mcpp_neighborhood"]
    )
)

# One row per CAD event for call-volume counting.
ranking_call_events = (
    ranking_calls
    .dropna(
        subset=[
            CALL_EVENT_ID_COLUMN,
            CALL_TIME_COLUMN,
        ]
    )
    .sort_values(CALL_TIME_COLUMN)
    .drop_duplicates(CALL_EVENT_ID_COLUMN)
    .copy()
)

# Shared coverage keeps all three displayed measures on the same dates.
response_history_start = (
    ranking_response["queued_time"]
    .dropna()
    .min()
    .normalize()
)
response_history_end = (
    ranking_response["queued_time"]
    .dropna()
    .max()
    .normalize()
)

call_history_start = (
    ranking_call_events[CALL_TIME_COLUMN]
    .dropna()
    .min()
    .normalize()
)
call_history_end = (
    ranking_call_events[CALL_TIME_COLUMN]
    .dropna()
    .max()
    .normalize()
)

crime_history_start = (
    ranking_crime["offense_date"]
    .dropna()
    .min()
    .normalize()
)
crime_history_end = (
    ranking_crime["offense_date"]
    .dropna()
    .max()
    .normalize()
)

ranking_history_start = max(
    response_history_start,
    call_history_start,
    crime_history_start,
)
ranking_history_end = min(
    response_history_end,
    call_history_end,
    crime_history_end,
)

ranking_analysis_start, ranking_analysis_end = (
    get_analysis_bounds(ranking_history_end)
)
ranking_analysis_start = max(
    ranking_analysis_start,
    ranking_history_start,
)

# QA: expose source neighborhood labels that do not match canonical MCPPs.
def _noncanonical_values(frame, column):
    values = set(
        frame[column]
        .dropna()
        .astype(str)
    )
    return sorted(
        values - canonical_ranking_set
    )

response_noncanonical = _noncanonical_values(
    ranking_response,
    "ranking_neighborhood",
)
call_noncanonical = _noncanonical_values(
    ranking_call_events,
    "ranking_neighborhood",
)
crime_noncanonical = _noncanonical_values(
    ranking_crime,
    "ranking_neighborhood",
)

print(
    "Shared ranking analysis domain:",
    ranking_analysis_start.date(),
    "→",
    ranking_analysis_end.date(),
)
print(
    "Response non-canonical labels:",
    response_noncanonical,
)
print(
    "Call non-canonical labels:",
    call_noncanonical,
)
print(
    "Crime non-canonical labels:",
    crime_noncanonical,
)


In [ ]:
RANKING_METRIC_OPTIONS = {
    "Response Time": "response_time",
    "Call Volume": "call_volume",
    "Crime Volume": "crime_volume",
}


def build_neighborhood_multimetric_ranking(
    response,
    call_events,
    crime,
    start_date,
    end_date,
    priorities,
    rank_metric="response_time",
    min_response_events=1,
    top_n=10,
):
    start = pd.to_datetime(start_date).normalize()
    end = pd.to_datetime(end_date).normalize()

    if end < start:
        raise ValueError(
            "end_date must be on or after start_date"
        )

    # ---------------------------------------------
    # RESPONSE TIME
    # ---------------------------------------------
    selected_response = response.loc[
        response["queued_time"]
        .dt.normalize()
        .between(start, end)
        & response["priority"].isin(priorities)
        & response["ranking_neighborhood"].isin(
            canonical_ranking_set
        )
    ].copy()

    response_summary = (
        selected_response
        .groupby(
            "ranking_neighborhood",
            as_index=False,
        )
        .agg(
            median_response_minutes=(
                "response_time_minutes",
                "median",
            ),
            qualified_response_events=(
                CALL_EVENT_ID_COLUMN,
                "nunique",
            ),
        )
    )

    # ---------------------------------------------
    # CALL VOLUME
    # ---------------------------------------------
    selected_calls = call_events.loc[
        call_events[CALL_TIME_COLUMN]
        .dt.normalize()
        .between(start, end)
        & call_events["ranking_neighborhood"].isin(
            canonical_ranking_set
        )
    ].copy()

    call_summary = (
        selected_calls
        .groupby(
            "ranking_neighborhood",
            as_index=False,
        )
        .agg(
            call_volume=(
                CALL_EVENT_ID_COLUMN,
                "nunique",
            ),
        )
    )

    # ---------------------------------------------
    # CRIME VOLUME
    # ---------------------------------------------
    selected_crime = crime.loc[
        crime["offense_date"]
        .dt.normalize()
        .between(start, end)
        & crime["ranking_neighborhood"].isin(
            canonical_ranking_set
        )
    ].copy()

    crime_summary = (
        selected_crime
        .groupby(
            "ranking_neighborhood",
            as_index=False,
        )
        .agg(
            crime_volume=(
                "offense_id",
                "nunique",
            ),
        )
    )

    # ---------------------------------------------
    # ONE ROW PER CANONICAL MCPP
    # ---------------------------------------------
    ranking = (
        canonical_ranking_neighborhoods
        .rename(
            columns={
                "mcpp_neighborhood":
                    "ranking_neighborhood"
            }
        )
        .merge(
            response_summary,
            on="ranking_neighborhood",
            how="left",
        )
        .merge(
            call_summary,
            on="ranking_neighborhood",
            how="left",
        )
        .merge(
            crime_summary,
            on="ranking_neighborhood",
            how="left",
        )
    )

    ranking["qualified_response_events"] = (
        ranking["qualified_response_events"]
        .fillna(0)
        .astype(int)
    )
    ranking["call_volume"] = (
        ranking["call_volume"]
        .fillna(0)
        .astype(int)
    )
    ranking["crime_volume"] = (
        ranking["crime_volume"]
        .fillna(0)
        .astype(int)
    )

    # ---------------------------------------------
    # RANKING ELIGIBILITY / ORDER
    # ---------------------------------------------
    if rank_metric == "response_time":
        ranking = ranking.loc[
            ranking["median_response_minutes"].notna()
            & (
                ranking["qualified_response_events"]
                >= min_response_events
            )
        ].copy()
        sort_column = "median_response_minutes"

    elif rank_metric == "call_volume":
        # All canonical MCPP neighborhoods remain eligible.
        # Zero-volume neighborhoods are retained so fullscreen
        # can show the complete 58-neighborhood table.
        sort_column = "call_volume"

    elif rank_metric == "crime_volume":
        # All canonical MCPP neighborhoods remain eligible.
        # Zero-volume neighborhoods are retained so fullscreen
        # can show the complete 58-neighborhood table.
        sort_column = "crime_volume"

    else:
        raise ValueError(
            f"Unknown rank_metric: {rank_metric}"
        )

    ranking = (
        ranking
        .sort_values(
            [
                sort_column,
                "ranking_neighborhood",
            ],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    if top_n is not None:
        ranking = (
            ranking
            .head(int(top_n))
            .reset_index(drop=True)
        )

    ranking["rank"] = ranking.index + 1

    return ranking


In [ ]:
def build_multimetric_ranking_rows(
    ranking,
    rank_metric,
):
    if ranking.empty:
        return [
            html.Div(
                "No neighborhoods meet the ranking requirements for this period.",
                style={
                    "padding": "18px",
                    "color": "#888888",
                    "fontSize": "12px",
                },
            )
        ]

    rows = []

    for row in ranking.itertuples():
        response_value = (
            "—"
            if pd.isna(
                row.median_response_minutes
            )
            else (
                f"{row.median_response_minutes:.1f} min"
            )
        )

        response_color = (
            "#ffffff"
            if rank_metric == "response_time"
            else "#dddddd"
        )
        call_color = (
            "#ffffff"
            if rank_metric == "call_volume"
            else "#dddddd"
        )
        crime_color = (
            "#ffffff"
            if rank_metric == "crime_volume"
            else "#dddddd"
        )

        rows.append(
            html.Div(
                children=[
                    html.Div(
                        str(row.rank),
                        style={
                            "color": "#888888",
                            "fontSize": "12px",
                            "fontWeight": "600",
                        },
                    ),

                    html.Div(
                        str(
                            row.ranking_neighborhood
                        ).title(),
                        style={
                            "color": "#ffffff",
                            "fontSize": "12px",
                            "fontWeight": "600",
                        },
                    ),

                    html.Div(
                        children=[
                            html.Div(
                                response_value,
                                style={
                                    "color": response_color,
                                    "fontSize": "13px",
                                    "fontWeight": (
                                        "700"
                                        if rank_metric
                                        == "response_time"
                                        else "600"
                                    ),
                                },
                            ),
                            html.Div(
                                (
                                    "n="
                                    f"{row.qualified_response_events:,}"
                                ),
                                style={
                                    "color": "#777777",
                                    "fontSize": "9px",
                                    "marginTop": "1px",
                                },
                            ),
                        ],
                        style={
                            "textAlign": "right",
                        },
                    ),

                    html.Div(
                        f"{row.call_volume:,}",
                        style={
                            "color": call_color,
                            "fontSize": "13px",
                            "fontWeight": (
                                "700"
                                if rank_metric
                                == "call_volume"
                                else "600"
                            ),
                            "textAlign": "right",
                        },
                    ),

                    html.Div(
                        f"{row.crime_volume:,}",
                        style={
                            "color": crime_color,
                            "fontSize": "13px",
                            "fontWeight": (
                                "700"
                                if rank_metric
                                == "crime_volume"
                                else "600"
                            ),
                            "textAlign": "right",
                        },
                    ),
                ],
                style={
                    "display": "grid",
                    "gridTemplateColumns": (
                        "45px minmax(170px, 1fr) "
                        "135px 105px 105px"
                    ),
                    "alignItems": "center",
                    "padding": "9px 14px",
                    "borderTop": (
                        "1px solid #2d2d2d"
                    ),
                },
            )
        )

    return rows


In [ ]:
RANKING_TABLE_BASE_STYLE = {
    "width": "100%",
    "maxWidth": "980px",
    "background": "#181818",
    "border": "1px solid #333333",
    "borderRadius": "8px",
    "overflow": "hidden",
    "boxSizing": "border-box",
    "position": "relative",
}

RANKING_TABLE_FULLSCREEN_STYLE = {
    **RANKING_TABLE_BASE_STYLE,
    "position": "fixed",
    "inset": "0",
    "zIndex": "10000",
    "width": "100vw",
    "maxWidth": "none",
    "height": "100vh",
    "borderRadius": "0",
    "overflowY": "auto",
}


neighborhood_ranking_table = html.Div(
    id="prototype-neighborhood-ranking-card",
    children=[
        # -------------------------------------------------
        # HEADER / LOCAL CONTROLS
        # -------------------------------------------------
        html.Div(
            children=[
                html.Div(
                    "Neighborhood Public-Safety KPIs Ranking",
                    style={
                        "color": "#bbbbbb",
                        "fontSize": "11px",
                        "fontWeight": "600",
                        "letterSpacing": "0.04em",
                        "textTransform": "uppercase",
                    },
                ),

                html.Div(
                    children=[
                        html.Div(
                            children=[
                                html.Span(
                                    "Rank by",
                                    style={
                                        "color": "#888888",
                                        "fontSize": "10px",
                                        "marginRight": "4px",
                                    },
                                ),
                                dcc.RadioItems(
                                    id=(
                                        "prototype-neighborhood-"
                                        "ranking-metric"
                                    ),
                                    options=[
                                        {
                                            "label": label,
                                            "value": metric,
                                        }
                                        for label, metric
                                        in RANKING_METRIC_OPTIONS.items()
                                    ],
                                    value="response_time",
                                    inline=True,
                                    inputStyle={
                                        "marginRight": "4px",
                                    },
                                    labelStyle={
                                        "marginLeft": "10px",
                                        "cursor": "pointer",
                                        "color": "#dddddd",
                                    },
                                    style={
                                        "fontSize": "10px",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                            ],
                            style={
                                "display": "flex",
                                "alignItems": "center",
                            },
                        ),

                        html.Button(
                            "↗",
                            id=(
                                "prototype-neighborhood-ranking-"
                                "fullscreen-button"
                            ),
                            n_clicks=0,
                            className="expand-button",
                            title="Expand neighborhood ranking",
                        ),
                    ],
                    style={
                        "display": "flex",
                        "alignItems": "center",
                        "gap": "12px",
                    },
                ),
            ],
            style={
                "display": "flex",
                "alignItems": "center",
                "justifyContent": "space-between",
                "padding": "12px 14px 7px",
                "gap": "16px",
            },
        ),

        html.Div(
            children=[
                dcc.RadioItems(
                    id=(
                        "prototype-response-ranking-"
                        "priority-scope"
                    ),
                    options=[
                        {
                            "label": label,
                            "value": label,
                        }
                        for label
                        in RESPONSE_PRIORITY_OPTIONS
                    ],
                    value="Priority 1–3",
                    inline=True,
                    inputStyle={
                        "marginRight": "4px",
                    },
                    labelStyle={
                        "marginRight": "14px",
                        "cursor": "pointer",
                        "color": "#dddddd",
                    },
                    style={
                        "fontSize": "10px",
                        "whiteSpace": "nowrap",
                    },
                ),

                html.Div(
                    children=[
                        html.Span(
                            "Min response events",
                            style={
                                "color": "#888888",
                                "fontSize": "10px",
                                "marginRight": "6px",
                            },
                        ),
                        dcc.Input(
                            id=(
                                "prototype-response-ranking-"
                                "min-events"
                            ),
                            type="number",
                            min=1,
                            step=1,
                            value=5,
                            debounce=True,
                            style={
                                "width": "55px",
                                "background": "#111111",
                                "color": "#dddddd",
                                "border": (
                                    "1px solid #444444"
                                ),
                                "borderRadius": "4px",
                                "padding": "3px 5px",
                                "fontSize": "10px",
                            },
                        ),
                    ],
                    style={
                        "display": "flex",
                        "alignItems": "center",
                    },
                ),
            ],
            style={
                "display": "flex",
                "alignItems": "center",
                "justifyContent": "space-between",
                "padding": "0 14px 10px",
                "gap": "16px",
                "flexWrap": "wrap",
            },
        ),

        # -------------------------------------------------
        # COLUMN HEADERS
        # -------------------------------------------------
        html.Div(
            children=[
                html.Div("Rank"),
                html.Div("Neighborhood"),
                html.Div(
                    "Median Response",
                    style={
                        "textAlign": "right",
                    },
                ),
                html.Div(
                    "Call Volume",
                    style={
                        "textAlign": "right",
                    },
                ),
                html.Div(
                    "Crime Volume",
                    style={
                        "textAlign": "right",
                    },
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": (
                    "45px minmax(170px, 1fr) "
                    "135px 105px 105px"
                ),
                "padding": "8px 14px",
                "borderTop": "1px solid #333333",
                "color": "#777777",
                "fontSize": "10px",
                "fontWeight": "600",
                "textTransform": "uppercase",
            },
        ),

        html.Div(
            id="prototype-neighborhood-ranking-body",
        ),

        html.Div(
            id="prototype-neighborhood-ranking-period-note",
            style={
                "padding": "9px 14px 11px",
                "borderTop": "1px solid #2d2d2d",
                "color": "#777777",
                "fontSize": "9px",
                "lineHeight": "13px",
            },
        ),
    ],
    style=RANKING_TABLE_BASE_STYLE,
)


In [ ]:
ranking_default_end = (
    ranking_analysis_end
    .date()
    .isoformat()
)

ranking_default_state = make_analysis_state(
    None,
    default_categories,
    [],
    [],
    ranking_default_end,
    ranking_default_end,
    default_categories,
)


neighborhood_ranking_app = Dash(
    __name__,
    assets_folder=str(REPO_ROOT / "assets"),
)

neighborhood_ranking_app.layout = html.Div(
    children=[
        dcc.Store(
            id="prototype-neighborhood-ranking-fullscreen",
            data=False,
        ),

        make_analysis_controls(
            ranking_default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            ranking_analysis_start,
            ranking_analysis_end,
        ),

        html.Div(
            neighborhood_ranking_table,
            style={
                "padding": "24px",
            },
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
    },
)


In [ ]:
@neighborhood_ranking_app.callback(
    Output(
        "prototype-neighborhood-ranking-fullscreen",
        "data",
    ),
    Input(
        "prototype-neighborhood-ranking-fullscreen-button",
        "n_clicks",
    ),
    State(
        "prototype-neighborhood-ranking-fullscreen",
        "data",
    ),
    prevent_initial_call=True,
)
def toggle_neighborhood_ranking_fullscreen(
    _n_clicks,
    is_fullscreen,
):
    return not bool(is_fullscreen)


@neighborhood_ranking_app.callback(
    Output(
        "prototype-neighborhood-ranking-card",
        "style",
    ),
    Output(
        "prototype-neighborhood-ranking-fullscreen-button",
        "children",
    ),
    Output(
        "prototype-neighborhood-ranking-fullscreen-button",
        "className",
    ),
    Output(
        "prototype-neighborhood-ranking-fullscreen-button",
        "title",
    ),
    Input(
        "prototype-neighborhood-ranking-fullscreen",
        "data",
    ),
)
def style_neighborhood_ranking_fullscreen(
    is_fullscreen,
):
    if is_fullscreen:
        return (
            RANKING_TABLE_FULLSCREEN_STYLE,
            "×",
            "close-fullscreen-button",
            "Close fullscreen view",
        )

    return (
        RANKING_TABLE_BASE_STYLE,
        "↗",
        "expand-button",
        "Expand neighborhood ranking",
    )


@neighborhood_ranking_app.callback(
    Output(
        "prototype-neighborhood-ranking-body",
        "children",
    ),
    Output(
        "prototype-neighborhood-ranking-period-note",
        "children",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "prototype-neighborhood-ranking-metric",
        "value",
    ),
    Input(
        "prototype-response-ranking-priority-scope",
        "value",
    ),
    Input(
        "prototype-response-ranking-min-events",
        "value",
    ),
    Input(
        "prototype-neighborhood-ranking-fullscreen",
        "data",
    ),
)
def update_neighborhood_multimetric_ranking(
    start_date,
    end_date,
    rank_metric,
    priority_scope_label,
    min_response_events,
    is_fullscreen,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        ranking_analysis_start,
        ranking_analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return (
            [
                html.Div(
                    "Invalid analysis period",
                    style={
                        "padding": "18px",
                        "color": "#888888",
                    },
                )
            ],
            "Shared crime/CAD period unavailable.",
        )

    current_start, current_end = bounded_dates

    priorities = RESPONSE_PRIORITY_OPTIONS[
        priority_scope_label
    ]

    if min_response_events is None:
        min_response_events = 1

    min_response_events = max(
        1,
        int(min_response_events),
    )

    ranking = build_neighborhood_multimetric_ranking(
        response=ranking_response,
        call_events=ranking_call_events,
        crime=ranking_crime,
        start_date=current_start,
        end_date=current_end,
        priorities=priorities,
        rank_metric=rank_metric,
        min_response_events=min_response_events,
        top_n=None if is_fullscreen else 10,
    )

    start_label = pd.to_datetime(
        current_start
    ).strftime("%b %d, %Y")
    end_label = pd.to_datetime(
        current_end
    ).strftime("%b %d, %Y")

    period_note = (
        f"Shared period: {start_label} – {end_label}. "
        "Response uses the selected priority scope; "
        "Call Volume counts distinct CAD events; "
        "Crime Volume counts distinct reported offenses. "
        "Crime type, subcategory, and neighborhood selections "
        "do not filter this figure."
    )

    return (
        build_multimetric_ranking_rows(
            ranking,
            rank_metric,
        ),
        period_note,
    )


In [ ]:
neighborhood_ranking_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8055,
)
